In [4]:
import xarray as xr
import numpy as np
import pandas as pd

# Open the downloaded netCDF
ds = xr.open_dataset("./era5_data/california_era5_daily.nc")

# ds will have dimensions: time, latitude, longitude
# Check variable names in ds.data_vars
print(ds)

<xarray.Dataset> Size: 298MB
Dimensions:          (time: 1826, lat: 101, lon: 101)
Coordinates:
  * lon              (lon) float64 808B -124.0 -123.9 -123.8 ... -114.1 -114.0
  * lat              (lat) float64 808B 42.0 41.9 41.8 41.7 ... 32.2 32.1 32.0
  * time             (time) datetime64[ns] 15kB 2019-01-01 ... 2023-12-31
Data variables:
    temperature      (time, lat, lon) float32 75MB ...
    dewpoint         (time, lat, lon) float32 75MB ...
    solar_radiation  (time, lat, lon) float32 75MB ...
    precipitation    (time, lat, lon) float32 75MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    institution:  European Centre for Medium-Range Weather Forecasts
    history:      Wed Feb 12 02:48:29 2025: cdo -f nc copy era5_data/extracte...
    CDO:          Climate Data Operators version 2.4.0 (https://mpimet.mpg.de...


In [5]:
# Convert to Pandas DataFrame
df_era5 = ds.to_dataframe().reset_index()  # This flattens time/lat/lon into rows
print("ERA5 DataFrame shape:", df_era5.shape)
print(df_era5.head())

ERA5 DataFrame shape: (18627026, 7)
        time   lat    lon  temperature    dewpoint  solar_radiation  \
0 2019-01-01  42.0 -124.0   274.066101  268.852936         130150.0   
1 2019-01-01  42.0 -123.9   272.343445  266.173248         128544.0   
2 2019-01-01  42.0 -123.8   271.009460  264.536530         126996.0   
3 2019-01-01  42.0 -123.7   270.855164  264.532623         125452.0   
4 2019-01-01  42.0 -123.6   270.454773  264.608795         123912.0   

   precipitation  
0            0.0  
1            0.0  
2            0.0  
3            0.0  
4            0.0  


In [6]:
# 1. Convert 'time' to just a date
df_era5['date'] = pd.to_datetime(df_era5['time']).dt.date

# 2. Round lat, lon to 2 decimals
df_era5['lat_rounded'] = df_era5['lat'].round(2)
df_era5['lon_rounded'] = df_era5['lon'].round(2)

# Keep only the columns we need
df_era5 = df_era5[['date','lat_rounded','lon_rounded','temperature','dewpoint','solar_radiation','precipitation']]

# Drop any rows with NaNs if needed (ERA5 might be complete though)
df_era5.dropna(subset=['temperature','dewpoint','solar_radiation','precipitation'], inplace=True)

print(df_era5.head())


         date  lat_rounded  lon_rounded  temperature    dewpoint  \
0  2019-01-01         42.0       -124.0   274.066101  268.852936   
1  2019-01-01         42.0       -123.9   272.343445  266.173248   
2  2019-01-01         42.0       -123.8   271.009460  264.536530   
3  2019-01-01         42.0       -123.7   270.855164  264.532623   
4  2019-01-01         42.0       -123.6   270.454773  264.608795   

   solar_radiation  precipitation  
0         130150.0            0.0  
1         128544.0            0.0  
2         126996.0            0.0  
3         125452.0            0.0  
4         123912.0            0.0  


In [7]:
df_fires = pd.read_csv("merged_final_rescaled.csv")

# Convert 'acq_date' to actual datetime, then to date
df_fires['acq_date'] = pd.to_datetime(df_fires['acq_date'], errors='coerce')
df_fires['date'] = df_fires['acq_date'].dt.date

# Round lat/long to match ERA5
df_fires['lat_rounded'] = df_fires['latitude'].round(2)
df_fires['lon_rounded'] = df_fires['longitude'].round(2)

print("Fires DataFrame shape:", df_fires.shape)
print(df_fires[['date','lat_rounded','lon_rounded','brightness','frp','NDVI']].head())


C:\Users\student\AppData\Local\Temp\ipykernel_25316\2091757408.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fires = pd.read_csv("merged_final_rescaled.csv")


Fires DataFrame shape: (1196228, 12)
         date  lat_rounded  lon_rounded  brightness   frp    NDVI
0  2019-01-01        40.70      -121.84      327.46  8.30  0.0253
1  2019-01-01        40.50      -124.25      327.22  3.39  0.0147
2  2019-01-01        39.39      -121.12      301.91  0.75  0.0124
3  2019-01-01        38.50      -120.68      315.35  1.66  0.0200
4  2019-01-01        38.32      -120.79      298.92  0.68  0.0286


In [8]:
# Merge them
df_merged = pd.merge(
    df_fires,
    df_era5,
    on=['date','lat_rounded','lon_rounded'],
    how='left'  # or 'inner' if you only want rows that match a weather record
)

print("Merged shape:", df_merged.shape)
print(df_merged.head())

# Save
df_merged.to_csv("merged_with_era5.csv", index=False)


Merged shape: (1196228, 16)
   latitude  longitude   acq_date  brightness   frp version  lat_round  \
0  40.70188 -121.84412 2019-01-01      327.46  8.30       2  40.700001   
1  40.50067 -124.25305 2019-01-01      327.22  3.39       2  40.500000   
2  39.38981 -121.12111 2019-01-01      301.91  0.75       2  39.389999   
3  38.50407 -120.68475 2019-01-01      315.35  1.66       2  38.500000   
4  38.32057 -120.79156 2019-01-01      298.92  0.68       2  38.320000   

    lon_round    NDVI        date  lat_rounded  lon_rounded  temperature  \
0 -121.839996  0.0253  2019-01-01        40.70      -121.84          NaN   
1 -124.250000  0.0147  2019-01-01        40.50      -124.25          NaN   
2 -121.120003  0.0124  2019-01-01        39.39      -121.12          NaN   
3 -120.680000  0.0200  2019-01-01        38.50      -120.68          NaN   
4 -120.790001  0.0286  2019-01-01        38.32      -120.79          NaN   

   dewpoint  solar_radiation  precipitation  
0       NaN             

In [9]:
print(df_merged[['temperature','dewpoint','solar_radiation','precipitation']].describe())


        temperature      dewpoint  solar_radiation  precipitation
count  11388.000000  11388.000000     1.138800e+04   1.138800e+04
mean     287.122528    276.793884     1.806965e+06   3.643493e-05
std        5.137839      5.958069     8.232663e+05   3.858515e-04
min      257.174500    248.939850     1.224600e+04   0.000000e+00
25%      283.971191    273.463867     1.187254e+06   0.000000e+00
50%      287.245361    277.262939     1.851516e+06   4.261732e-07
75%      290.450684    280.904846     2.377460e+06   8.583069e-07
max      305.968506    293.734131     3.889472e+06   1.540198e-02


In [10]:
import pandas as pd

# Replace 'df_merged' with the actual name of your merged dataset DataFrame
# e.g., df_merged = pd.read_csv("merged_with_era5.csv")
# for this example, we'll assume you already have df_merged in memory.

#####################
# 1. Row Count
#####################
print("=== 1) Final Merged Dataset Shape ===")
print(df_merged.shape)

#####################
# 2. Missing Values
#####################
print("\n=== 2) Missing Values per Column ===")
print(df_merged.isnull().sum())

#####################
# 3. Date Range
#####################
print("\n=== 3) Date Range ===")
if 'date' in df_merged.columns:
    print("Min date:", df_merged['date'].min())
    print("Max date:", df_merged['date'].max())
else:
    print("No 'date' column found. Please ensure you created it.")

#####################
# 4. Descriptive Stats for Weather
#####################
weather_cols = ['temperature', 'dewpoint', 'solar_radiation', 'precipitation']
existing_weather_cols = [col for col in weather_cols if col in df_merged.columns]

if existing_weather_cols:
    print("\n=== 4) Descriptive Stats for Weather Columns ===")
    print(df_merged[existing_weather_cols].describe())
else:
    print("\nNo recognized weather columns in the DataFrame.")

#####################
# 5. Convert Units (Optional)
#####################
# If temperature is in Kelvin (typical ERA5), convert to Celsius
# If precipitation is in meters/day, convert to mm/day
if 'temperature' in df_merged.columns:
    df_merged['temp_c'] = df_merged['temperature'] - 273.15
if 'precipitation' in df_merged.columns:
    df_merged['precip_mm'] = df_merged['precipitation'] * 1000.0

#####################
# 6. Stats after Conversion & NaNs
#####################
# Example stats on converted columns
convert_cols = ['temp_c', 'precip_mm']
existing_convert_cols = [col for col in convert_cols if col in df_merged.columns]

if existing_convert_cols:
    print("\n=== 6) Descriptive Stats for Converted Columns ===")
    print(df_merged[existing_convert_cols].describe())

    print("\nNaN Count in Converted Columns:")
    print(df_merged[existing_convert_cols].isnull().sum())


=== 1) Final Merged Dataset Shape ===
(1196228, 16)

=== 2) Missing Values per Column ===
latitude                 0
longitude                0
acq_date                 0
brightness               0
frp                      0
version                  0
lat_round                0
lon_round                0
NDVI                     0
date                     0
lat_rounded              0
lon_rounded              0
temperature        1184840
dewpoint           1184840
solar_radiation    1184840
precipitation      1184840
dtype: int64

=== 3) Date Range ===
Min date: 2019-01-01
Max date: 2023-12-27

=== 4) Descriptive Stats for Weather Columns ===
        temperature      dewpoint  solar_radiation  precipitation
count  11388.000000  11388.000000     1.138800e+04   1.138800e+04
mean     287.122528    276.793884     1.806965e+06   3.643493e-05
std        5.137839      5.958069     8.232663e+05   3.858515e-04
min      257.174500    248.939850     1.224600e+04   0.000000e+00
25%      283.971191 

In [11]:
import pandas as pd

# We assume you already have:
# df_fires (with 'lat_rounded','lon_rounded','date')
# df_era5  (with 'lat_rounded','lon_rounded','date')

unique_fires_lats = set(df_fires['lat_rounded'].unique())
unique_fires_lons = set(df_fires['lon_rounded'].unique())

unique_era5_lats = set(df_era5['lat_rounded'].unique())
unique_era5_lons = set(df_era5['lon_rounded'].unique())

common_lats = unique_fires_lats.intersection(unique_era5_lats)
common_lons = unique_fires_lons.intersection(unique_era5_lons)

print("Unique lat_rounded in Fire Data:", len(unique_fires_lats))
print("Unique lat_rounded in ERA5:", len(unique_era5_lats))
print("Lat_rounded in common:", len(common_lats))

print("Unique lon_rounded in Fire Data:", len(unique_fires_lons))
print("Unique lon_rounded in ERA5:", len(unique_era5_lons))
print("Lon_rounded in common:", len(common_lons))


Unique lat_rounded in Fire Data: 949
Unique lat_rounded in ERA5: 101
Lat_rounded in common: 95
Unique lon_rounded in Fire Data: 1024
Unique lon_rounded in ERA5: 101
Lon_rounded in common: 99


In [12]:
df_fires['lat_rounded'] = df_fires['latitude'].round(1)
df_fires['lon_rounded'] = df_fires['longitude'].round(1)

df_era5['lat_rounded'] = df_era5['lat'].round(1)
df_era5['lon_rounded'] = df_era5['lon'].round(1)


KeyError: 'lat'

In [13]:
print(df_era5.columns)


Index(['date', 'lat_rounded', 'lon_rounded', 'temperature', 'dewpoint',
       'solar_radiation', 'precipitation'],
      dtype='object')


In [14]:
df_era5['lat_rounded'] = df_era5['lat_rounded'].round(1)
df_era5['lon_rounded'] = df_era5['lon_rounded'].round(1)


In [15]:
df_fires['lat_rounded'] = df_fires['latitude'].round(1)
df_fires['lon_rounded'] = df_fires['longitude'].round(1)


In [16]:
df_merged_1dec = pd.merge(
    df_fires,
    df_era5[['date','lat_rounded','lon_rounded','temperature','dewpoint','solar_radiation','precipitation']],
    on=['date','lat_rounded','lon_rounded'],
    how='left'
)


In [17]:
print("ERA5 lat_rounded unique:", df_era5['lat_rounded'].nunique())
print("Fires lat_rounded unique:", df_fires['lat_rounded'].nunique())


ERA5 lat_rounded unique: 101
Fires lat_rounded unique: 96


In [18]:
df_merged = pd.merge(
    df_fires,
    df_era5[['date','lat_rounded','lon_rounded','temperature','dewpoint','solar_radiation','precipitation']],
    on=['date','lat_rounded','lon_rounded'],
    how='left'
)

print("Merged shape:", df_merged.shape)

# Count how many rows have actual (non-NaN) weather data:
weather_cols = ['temperature','dewpoint','solar_radiation','precipitation']
print("Non-NaN in Weather Columns:")
print(df_merged[weather_cols].notnull().all(axis=1).sum())

# Or you can see how many are NaN:
print("NaN in Weather Columns:")
print(df_merged[weather_cols].isnull().sum())


Merged shape: (1196228, 16)
Non-NaN in Weather Columns:
1190114
NaN in Weather Columns:
temperature        6114
dewpoint           6114
solar_radiation    6114
precipitation      6114
dtype: int64


In [19]:
df_final = df_merged.dropna(subset=['temperature','dewpoint','solar_radiation','precipitation'])
print(df_final.shape)  # Expect ~1,190,114


(1190114, 16)


In [20]:
df_final['temp_c'] = df_final['temperature'] - 273.15


C:\Users\student\AppData\Local\Temp\ipykernel_25316\3911060508.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['temp_c'] = df_final['temperature'] - 273.15


In [21]:
df_final['precip_mm'] = df_final['precipitation'] * 1000


C:\Users\student\AppData\Local\Temp\ipykernel_25316\1718029764.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['precip_mm'] = df_final['precipitation'] * 1000


In [22]:
# Convert acq_date to a proper datetime (if not done already)
df_final['acq_date'] = pd.to_datetime(df_final['acq_date'], errors='coerce')

# Create the time-based columns
df_final['day_of_year'] = df_final['acq_date'].dt.dayofyear
df_final['month'] = df_final['acq_date'].dt.month
df_final['year'] = df_final['acq_date'].dt.year

C:\Users\student\AppData\Local\Temp\ipykernel_25316\1821161468.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['acq_date'] = pd.to_datetime(df_final['acq_date'], errors='coerce')
C:\Users\student\AppData\Local\Temp\ipykernel_25316\1821161468.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['day_of_year'] = df_final['acq_date'].dt.dayofyear
C:\Users\student\AppData\Local\Temp\ipykernel_25316\1821161468.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fr

In [25]:
df_final.to_csv("final.csv", index=False)

In [26]:


# Now you can safely include 'day_of_year','month','year' in your features
features = [
    'latitude', 'longitude', 'NDVI',
    'day_of_year','month','year',
    'temp_c','dewpoint','solar_radiation','precip_mm'
]
X = df_final[features]
y = df_final['brightness']


In [27]:
features = [
    'latitude', 'longitude', 'NDVI',
    'temp_c','dewpoint','solar_radiation','precip_mm'
]
X = df_final[features]
y = df_final['brightness']


In [28]:
print(df_final.isnull().sum())

latitude           0
longitude          0
acq_date           0
brightness         0
frp                0
version            0
lat_round          0
lon_round          0
NDVI               0
date               0
lat_rounded        0
lon_rounded        0
temperature        0
dewpoint           0
solar_radiation    0
precipitation      0
temp_c             0
precip_mm          0
day_of_year        0
month              0
year               0
dtype: int64


In [29]:
# Example feature set
features = [
    'latitude', 'longitude', 'NDVI',
    'day_of_year', 'month', 'year',
    'temp_c', 'dewpoint', 'solar_radiation', 'precip_mm'
]
X = df_final[features]
y = df_final['brightness']


In [30]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# 1. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Model Training
rf = RandomForestRegressor(
    n_estimators=100, 
    max_depth=10, 
    random_state=42
)
rf.fit(X_train, y_train)

# 3. Predictions & Metrics
y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R^2:", r2)


KeyboardInterrupt: 